# CUDA Tic-Tac-Toe — Jupyter Notebook Study Manual

## Learning objectives

By the end of this notebook, you should be able to:

- Explain how CUDA threads execute in parallel.
- Understand CUDA kernels, blocks, grids, and thread indices.
- Transfer data between CPU and GPU.
- Implement a CUDA kernel that checks Tic-Tac-Toe winning lines.
- Understand synchronization and GPU result retrieval.
- Explain why a tiny Tic-Tac-Toe game is mainly an educational CUDA example rather than a performance optimization.

> **Requirement:** This notebook must run in an environment with an NVIDIA GPU and the CUDA compiler (`nvcc`).

## 1. Tic-Tac-Toe board representation

We represent the 3×3 board as a one-dimensional array of 9 positions:

```text
 0 | 1 | 2
---+---+---
 3 | 4 | 5
---+---+---
 6 | 7 | 8
```

The eight possible winning lines are:

```text
Rows       Columns       Diagonals

0 1 2      0 3 6        0 4 8
3 4 5      1 4 7        2 4 6
6 7 8      2 5 8
```

The key CUDA idea is simple:

**One CUDA thread checks one winning line.**

Therefore, 8 CUDA threads can check all 8 winning possibilities in parallel.

In [ ]:
# Check that the notebook is running in a CUDA-capable environment.
!nvcc --version

## 2. CPU vs GPU responsibility

The game is divided into two parts:

| Task | CPU | GPU |
|---|---|---|
| Read player's move | ✓ | |
| Update board | ✓ | |
| Check winning lines | | ✓ |
| Return winning-line results | | ✓ → CPU |
| Display board | ✓ | |
| Switch player | ✓ | |

The CPU controls the game loop, while the GPU performs the parallel winner checks.

## 3. CUDA programming concepts

### `__global__`

A function marked with `__global__` is a CUDA kernel. It is called by the CPU and executed by GPU threads.

```cpp
__global__ void checkWinner(...)
{
    ...
}
```

### Thread index

CUDA provides built-in variables such as:

```cpp
threadIdx.x
blockIdx.x
blockDim.x
```

For this game we launch:

```cpp
checkWinner<<<1, 8>>>(...);
```

This means:

- 1 block
- 8 threads
- Thread 0 checks winning line 0
- Thread 1 checks winning line 1
- ...
- Thread 7 checks winning line 7

The global thread index is:

```cpp
int line = blockIdx.x * blockDim.x + threadIdx.x;
```

## 4. The CUDA kernel

The following kernel checks all eight possible winning lines.

Each thread reads three board positions and determines whether they contain the same non-empty symbol.

Result encoding:

- `0` → no winner on this line
- `1` → X wins on this line
- `2` → O wins on this line

In [ ]:
%%writefile tictactoe.cu
#include <stdio.h>
#include <cuda_runtime.h>

// --------------------------------------------------
// CUDA Kernel
// Each thread checks one possible winning line
// --------------------------------------------------

__global__ void checkWinner(char *board, int *results)
{
    int line = blockIdx.x * blockDim.x + threadIdx.x;

    if (line >= 8)
        return;

    int winningLines[8][3] =
    {
        {0, 1, 2},   // Row 1
        {3, 4, 5},   // Row 2
        {6, 7, 8},   // Row 3

        {0, 3, 6},   // Column 1
        {1, 4, 7},   // Column 2
        {2, 5, 8},   // Column 3

        {0, 4, 8},   // Diagonal 1
        {2, 4, 6}    // Diagonal 2
    };

    int a = winningLines[line][0];
    int b = winningLines[line][1];
    int c = winningLines[line][2];

    if (board[a] != ' ' &&
        board[a] == board[b] &&
        board[b] == board[c])
    {
        if (board[a] == 'X')
            results[line] = 1;
        else if (board[a] == 'O')
            results[line] = 2;
    }
}

// --------------------------------------------------
// Display board
// --------------------------------------------------

void displayBoard(char board[])
{
    printf("\n");

    printf(" %c | %c | %c\n", board[0], board[1], board[2]);
    printf("---+---+---\n");
    printf(" %c | %c | %c\n", board[3], board[4], board[5]);
    printf("---+---+---\n");
    printf(" %c | %c | %c\n", board[6], board[7], board[8]);

    printf("\n");
}

// --------------------------------------------------
// Check whether board is full
// --------------------------------------------------

bool boardFull(char board[])
{
    for (int i = 0; i < 9; i++)
    {
        if (board[i] == ' ')
            return false;
    }

    return true;
}

// --------------------------------------------------
// Main
// --------------------------------------------------

int main()
{
    char board[9] =
    {
        ' ', ' ', ' ',
        ' ', ' ', ' ',
        ' ', ' ', ' '
    };

    char *d_board;
    int *d_results;

    int results[8];

    // Allocate GPU memory
    cudaMalloc((void**)&d_board, 9 * sizeof(char));
    cudaMalloc((void**)&d_results, 8 * sizeof(int));

    char player = 'X';
    int position;

    printf("=================================\n");
    printf("       CUDA TIC-TAC-TOE\n");
    printf("=================================\n");

    printf("\nPositions are:\n");
    printf(" 1 | 2 | 3\n");
    printf("---+---+---\n");
    printf(" 4 | 5 | 6\n");
    printf("---+---+---\n");
    printf(" 7 | 8 | 9\n");

    while (true)
    {
        displayBoard(board);

        printf("Player %c, enter position (1-9): ", player);
        scanf("%d", &position);

        position--;

        if (position < 0 || position > 8)
        {
            printf("Invalid position!\n");
            continue;
        }

        if (board[position] != ' ')
        {
            printf("Position already occupied!\n");
            continue;
        }

        board[position] = player;

        // CPU -> GPU
        cudaMemcpy(
            d_board,
            board,
            9 * sizeof(char),
            cudaMemcpyHostToDevice
        );

        // Reset previous results
        cudaMemset(
            d_results,
            0,
            8 * sizeof(int)
        );

        // Launch 8 CUDA threads.
        // Each thread checks one winning line.
        checkWinner<<<1, 8>>>(
            d_board,
            d_results
        );

        // Wait for kernel completion
        cudaDeviceSynchronize();

        // GPU -> CPU
        cudaMemcpy(
            results,
            d_results,
            8 * sizeof(int),
            cudaMemcpyDeviceToHost
        );

        bool winnerFound = false;

        for (int i = 0; i < 8; i++)
        {
            if (results[i] == 1)
            {
                displayBoard(board);
                printf("Player X wins!\n");
                winnerFound = true;
                break;
            }

            if (results[i] == 2)
            {
                displayBoard(board);
                printf("Player O wins!\n");
                winnerFound = true;
                break;
            }
        }

        if (winnerFound)
            break;

        if (boardFull(board))
        {
            displayBoard(board);
            printf("Game Draw!\n");
            break;
        }

        if (player == 'X')
            player = 'O';
        else
            player = 'X';
    }

    // Free GPU memory
    cudaFree(d_board);
    cudaFree(d_results);

    return 0;
}


## 5. Compile the CUDA program

Run the following command to compile the `.cu` source file:

In [ ]:
!nvcc tictactoe.cu -o tictactoe

## 6. Run the game

On Linux/WSL:

```bash
./tictactoe
```

On Windows:

```bash
tictactoe.exe
```

The notebook cell below runs the Linux/WSL executable.

In [ ]:
!./tictactoe

## 7. Understanding the kernel step by step

The kernel begins with:

```cpp
__global__ void checkWinner(char *board, int *results)
```

`board` is a pointer to GPU memory containing the 9 board cells.

`results` is GPU memory containing 8 integer results.

### Step 1 — identify the thread

```cpp
int line = blockIdx.x * blockDim.x + threadIdx.x;
```

For:

```text
checkWinner<<<1, 8>>>
```

the thread IDs are:

```text
Thread 0 → line 0
Thread 1 → line 1
Thread 2 → line 2
Thread 3 → line 3
Thread 4 → line 4
Thread 5 → line 5
Thread 6 → line 6
Thread 7 → line 7
```

### Step 2 — select the winning line

For example:

```cpp
{0, 1, 2}
```

means the thread checks the first row.

```cpp
{0, 4, 8}
```

means the thread checks the first diagonal.

### Step 3 — compare the three cells

```cpp
if (board[a] != ' ' &&
    board[a] == board[b] &&
    board[b] == board[c])
```

This means:

1. The first cell must not be empty.
2. First and second cells must match.
3. Second and third cells must match.

Therefore all three cells contain `X` or all three contain `O`.

## 8. CPU → GPU → CPU data movement

The game repeatedly performs three important operations.

### CPU → GPU

```cpp
cudaMemcpy(
    d_board,
    board,
    9 * sizeof(char),
    cudaMemcpyHostToDevice
);
```

The current board is copied from CPU memory to GPU memory.

### GPU computation

```cpp
checkWinner<<<1, 8>>>(d_board, d_results);
```

Eight threads check the eight possible winning lines.

### GPU → CPU

```cpp
cudaMemcpy(
    results,
    d_results,
    8 * sizeof(int),
    cudaMemcpyDeviceToHost
);
```

The results are copied back so the CPU can decide whether the game has ended.

## 9. Synchronization

The program uses:

```cpp
cudaDeviceSynchronize();
```

This makes the CPU wait until the GPU kernel has finished.

Conceptually:

```text
CPU
 │
 │ launch kernel
 ▼
GPU
 │
 │ check 8 winning lines
 ▼
Kernel finished
 │
 ▼
CPU continues
```

Without appropriate synchronization or a synchronizing operation, you must be careful not to assume asynchronous GPU work has completed before using its results.

## 10. Memory management

### Allocate GPU memory

```cpp
cudaMalloc((void**)&d_board, 9 * sizeof(char));
cudaMalloc((void**)&d_results, 8 * sizeof(int));
```

### Copy data to GPU

```cpp
cudaMemcpy(..., cudaMemcpyHostToDevice);
```

### Copy data back

```cpp
cudaMemcpy(..., cudaMemcpyDeviceToHost);
```

### Release GPU memory

```cpp
cudaFree(d_board);
cudaFree(d_results);
```

The basic CUDA memory lifecycle is:

```text
cudaMalloc
    ↓
cudaMemcpy CPU → GPU
    ↓
Kernel execution
    ↓
cudaMemcpy GPU → CPU
    ↓
cudaFree
```

## 11. Why use CUDA for Tic-Tac-Toe?

A 3×3 Tic-Tac-Toe board is very small. Running it on a GPU will normally be slower than simply checking the eight lines on the CPU because:

- Kernel launches have overhead.
- CPU–GPU memory transfers have overhead.
- Only eight small checks are being performed.

So this project should be understood as a **CUDA learning exercise**, not a practical GPU optimization.

The educational lesson is:

> If independent pieces of work can be assigned to different threads, CUDA can execute those pieces concurrently.

For a much larger game-state search problem, the same idea becomes more useful.

## 12. Experiment: change the number of threads

The current kernel uses:

```cpp
checkWinner<<<1, 8>>>(d_board, d_results);
```

Try changing the launch configuration to:

```cpp
checkWinner<<<1, 16>>>(d_board, d_results);
```

The kernel contains:

```cpp
if (line >= 8)
    return;
```

so threads 8–15 immediately stop.

**Question:** Why does this still produce the correct answer?

**Answer:** Only threads with `line < 8` correspond to valid winning lines. Extra threads return without modifying the results.

## 13. Exercise 1 — Trace the threads

Given:

```cpp
checkWinner<<<1, 8>>>(d_board, d_results);
```

Complete this table:

| Thread | Winning line |
|---:|---|
| 0 | ? |
| 1 | ? |
| 2 | ? |
| 3 | ? |
| 4 | ? |
| 5 | ? |
| 6 | ? |
| 7 | ? |

Expected mapping:

```text
0 → {0,1,2}
1 → {3,4,5}
2 → {6,7,8}
3 → {0,3,6}
4 → {1,4,7}
5 → {2,5,8}
6 → {0,4,8}
7 → {2,4,6}
```

## 14. Exercise 2 — Matrix-style thinking

The same parallel idea can be used for matrix operations.

For matrix addition:

```text
A[i][j] + B[i][j] → C[i][j]
```

One CUDA thread can calculate one output element.

For Tic-Tac-Toe:

```text
winning line → one CUDA thread
```

This gives a useful general CUDA pattern:

```text
Independent task
       ↓
One CUDA thread
       ↓
Parallel execution
```

## 15. Exercise 3 — Improve the program

Try these modifications:

1. Add input validation so non-numeric input does not terminate the game.
2. Add a function to print which winning line was detected.
3. Add CUDA error checking after `cudaMalloc`, `cudaMemcpy`, and kernel execution.
4. Replace the fixed `8` with a named constant such as `NUM_LINES`.
5. Create a separate CUDA kernel to determine whether the board is full.
6. Create a GPU-based AI that evaluates possible moves.

### Challenge

Design a GPU Tic-Tac-Toe AI where different CUDA threads evaluate different possible game moves.

For example:

```text
Thread 0 → evaluate move 1
Thread 1 → evaluate move 2
Thread 2 → evaluate move 3
...
Thread 8 → evaluate move 9
```

A more advanced version can evaluate many future game states simultaneously.

# Summary

You have implemented a complete CUDA Tic-Tac-Toe game using:

- CUDA C++
- CUDA kernel
- `__global__`
- `threadIdx.x`
- `blockIdx.x`
- `blockDim.x`
- `cudaMalloc`
- `cudaMemcpy`
- `cudaMemset`
- `cudaDeviceSynchronize`
- `cudaFree`
- CPU–GPU data transfer
- Parallel winner checking

The central idea is:

```text
Tic-Tac-Toe
    ↓
8 possible winning lines
    ↓
8 independent checks
    ↓
8 CUDA threads
    ↓
Parallel execution
```

This same principle can be extended to much larger problems such as matrix operations, image processing, simulations, search, machine learning, and large-scale game-state evaluation.